In [27]:
import boto3
from dotenv import load_dotenv
import os
import sqlalchemy
import pymysql

load_dotenv()

aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')
region_name = os.getenv('region_name')
master_user_password = os.getenv('master_db_password')
master_username = os.getenv('master_db_name')
port = os.getenv('port') 

# Créez une session boto3
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name
)

# Créez un client RDS
rds_client = session.client('rds')

# Paramètres de la base de données
db_instance_identifier = 'dbkayak'
db_instance_class = 'db.t4g.micro'  # Free Tier instance type
engine = 'mysql'
allocated_storage = 20  # Free Tier allows up to 20 GB

In [29]:
# Vérifiez si l'instance RDS existe déjà
try:
    db_instance_info = rds_client.describe_db_instances(DBInstanceIdentifier=db_instance_identifier)
    print(f"L'instance RDS '{db_instance_identifier}' existe déjà.")
except rds_client.exceptions.DBInstanceNotFoundFault:
    # Créez la base de données RDS si elle n'existe pas
    try:
        response = rds_client.create_db_instance(
            DBInstanceIdentifier=db_instance_identifier,
            DBInstanceClass=db_instance_class,
            Engine=engine,
            MasterUsername=master_username,
            MasterUserPassword=master_user_password,
            AllocatedStorage=allocated_storage,
            BackupRetentionPeriod=7,  # Number of days to retain backups
            MultiAZ=False,  # Free Tier does not support Multi-AZ deployments
            PubliclyAccessible=True,  # Set to False if you don't want the DB to be publicly accessible
            StorageType='gp2',  # General Purpose SSD
        )
        print("Creating RDS instance...")
        print(response)
    except Exception as e:
        print(f"Error creating RDS instance: {e}")

L'instance RDS 'dbkayak' existe déjà.


In [30]:
# Obtenez les informations de l'instance RDS
try:
    db_instance_info = rds_client.describe_db_instances(DBInstanceIdentifier=db_instance_identifier)
    endpoint = db_instance_info['DBInstances'][0]['Endpoint']['Address']
    port = db_instance_info['DBInstances'][0]['Endpoint']['Port']
    name=db_instance_info['DBInstances'][0]['DBInstanceIdentifier']
    print(f"Name: {name}")  
    print(f"Endpoint: {endpoint}")
    print(f"Port: {port}")
except Exception as e:
    print(f"Error retrieving DB instance info: {e}")

Name: dbkayak
Endpoint: dbkayak.c3g8yqkisjyz.eu-west-3.rds.amazonaws.com
Port: 3306


In [31]:
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError

new_database_name = 'dbkayak'

# Chaîne de connexion sans base de données pour la création de la base
base_connection_string = f"mysql+pymysql://{master_username}:{master_user_password}@{endpoint}:{port}"
database_connection_string = f"mysql+pymysql://{master_username}:{master_user_password}@{endpoint}:{port}/{new_database_name}"

# Créer le moteur SQLAlchemy sans base de données spécifique
base_engine = create_engine(base_connection_string)

def check_and_create_database(engine, db_name):
    """Vérifie si la base de données existe et la crée si elle n'existe pas."""
    database_exists_query = text(f"SELECT SCHEMA_NAME FROM INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME = :db_name")

    try:
        with engine.connect() as connection:
            # Vérifie si la base de données existe
            result = connection.execute(database_exists_query, {"db_name": db_name}).fetchone()
            if result:
                print(f"La base de données '{db_name}' existe déjà.")
            else:
                # Crée la base de données si elle n'existe pas
                connection.execute(text(f"CREATE DATABASE {db_name}"))
                print(f"Base de données '{db_name}' créée avec succès.")
    except OperationalError as e:
        print(f"Erreur lors de la vérification ou de la création de la base de données : {e}")

def connect_to_database(connection_url):
    """Essaie de se connecter à une base de données et gère les erreurs de connexion."""
    engine = create_engine(connection_url)
    try:
        with engine.connect() as connection:
            print("Connexion réussie à la base de données MySQL!")
    except OperationalError as e:
        print(f"Erreur lors de la connexion à la base de données : {e}")

# Vérifiez et créez la base de données si nécessaire
check_and_create_database(base_engine, new_database_name)

# Créer le moteur pour se connecter à la nouvelle base de données
connect_to_database(database_connection_string)


Base de données 'dbkayak' créée avec succès.
Connexion réussie à la base de données MySQL!


In [32]:
import pandas as pd
import requests

In [33]:
url='https://tmopenlabbucket.s3.eu-west-3.amazonaws.com/City_Meteo_Rank_Booking.csv'

df = pd.read_csv(url,index_col=0)

In [34]:
df.head(10)

,city,lat,lon,CCM,name,url,score,description,latitude,longitude
0,Le Havre,49.493898,0.107973,0.927,"The Originals Boutique, Hôtel d'Angleterre, Le...",https://www.booking.com/hotel/fr/comfort-d-ang...,7.6,This hotel is located in the town centre of Le...,49.494049,0.099366
1,Le Havre,49.493898,0.107973,0.927,LA PARENTHÈSE HAVRAISE - Parking privé Plein c...,https://www.booking.com/hotel/fr/la-parenthese...,9.1,LA PARENTHÈSE HAVRAISE - Parking privé Plein c...,49.496836,0.106893
2,Le Havre,49.493898,0.107973,0.927,Best Western ARThotel,https://www.booking.com/hotel/fr/art.en-gb.htm...,8.0,The Best Western Art Hotel is located in the h...,49.491194,0.106461
3,Le Havre,49.493898,0.107973,0.927,Aparthotel Adagio Access Le Havre Les Docks,https://www.booking.com/hotel/fr/adagio-access...,8.7,"Located in Le Havre, Aparthotel Adagio Access ...",49.487419,0.131511
4,Le Havre,49.493898,0.107973,0.927,Best Western Plus Le Havre Centre Gare,https://www.booking.com/hotel/fr/hotelterminus...,8.3,NaN,49.493344,0.124318
5,Le Havre,49.493898,0.107973,0.927,Hotel de Charme La Bonne Adresse - Logis hotels,https://www.booking.com/hotel/fr/des-phares.en...,8.4,"Just 150 metres from Le Havre's beaches, Hotel...",49.503189,0.088821
6,Le Havre,49.493898,0.107973,0.927,"Hôtel Richelieu, Le Havre Centre-Ville, Perret",https://www.booking.com/hotel/fr/le-richelieu-...,8.6,Located between the town hall and the cathedra...,49.488811,0.107602
7,Le Havre,49.493898,0.107973,0.927,Appart Hotel Odalys City Le Havre Centre Les D...,https://www.booking.com/hotel/fr/odalys-city-l...,8.5,NaN,49.492504,0.129779
8,Le Havre,49.493898,0.107973,0.927,All Suites Appart Hôtel - Le Havre Centre - Le...,https://www.booking.com/hotel/fr/all-suites-ap...,8.5,NaN,49.488691,0.125929
9,Le Havre,49.493898,0.107973,0.927,"""L'amarrage"" 2 chambres Perret Pleine Vue Mer",https://www.booking.com/hotel/fr/appart-perret...,9.6,"Offering a casino and sea view, ""L'amarrage"" 2...",49.486642,0.111221


In [35]:
engine = create_engine(database_connection_string)

In [36]:
# Insérez le DataFrame dans la base de données MySQL
try:
    df.to_sql(name='dbkayak', con=engine, if_exists='replace', index=False)
    print("DataFrame inséré avec succès dans la table 'dbkayak' de la base de données MySQL.")
except Exception as e:
    print(f"Erreur lors de l'insertion du DataFrame dans la base de données: {e}")

DataFrame inséré avec succès dans la table 'dbkayak' de la base de données MySQL.


In [37]:
from sqlalchemy import text

stmt = text("SELECT * FROM dbkayak.dbkayak LIMIT 5")

df = pd.read_sql_query(con=engine.connect(), sql=stmt)

df

,city,lat,lon,CCM,name,url,score,description,latitude,longitude
0,Le Havre,49.493898,0.107973,0.927,"The Originals Boutique, Hôtel d'Angleterre, Le...",https://www.booking.com/hotel/fr/comfort-d-ang...,7.6,This hotel is located in the town centre of Le...,49.494049,0.099366
1,Le Havre,49.493898,0.107973,0.927,LA PARENTHÈSE HAVRAISE - Parking privé Plein c...,https://www.booking.com/hotel/fr/la-parenthese...,9.1,LA PARENTHÈSE HAVRAISE - Parking privé Plein c...,49.496836,0.106893
2,Le Havre,49.493898,0.107973,0.927,Best Western ARThotel,https://www.booking.com/hotel/fr/art.en-gb.htm...,8.0,The Best Western Art Hotel is located in the h...,49.491194,0.106461
3,Le Havre,49.493898,0.107973,0.927,Aparthotel Adagio Access Le Havre Les Docks,https://www.booking.com/hotel/fr/adagio-access...,8.7,"Located in Le Havre, Aparthotel Adagio Access ...",49.487419,0.131511
4,Le Havre,49.493898,0.107973,0.927,Best Western Plus Le Havre Centre Gare,https://www.booking.com/hotel/fr/hotelterminus...,8.3,None,49.493344,0.124318
